In [1]:
import os
os.chdir(r"D:\doc\2026\FINAL_CODE\src")
print(os.getcwd())

D:\doc\2026\FINAL_CODE\src


In [2]:
import codeStat.modelimports.modelFunction as func
from codeStat.modelimports.modelFunction import *
import librosa
import glob
import pandas as pd

In [3]:
dir_save_path="D:/doc/2026/FINAL_CODE/src/codeStat/modelling/voiceModelling/srcVoiceModelling/GMM_TDNN/nonfix_out/GMM"

In [4]:
# Load CSV
df_csv = pd.read_csv("D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/samplingStratified/out_samplingStratified/dataSampledSmall.csv")

# Buat dictionary
dict_transkrip = {
    os.path.splitext(os.path.basename(path))[0]: kalimat
    for path, kalimat in zip(df_csv["Path_Data"], df_csv["sentences"])
}

def baca_transkrip(wav_path: str) -> str:
    audio_id = os.path.splitext(os.path.basename(wav_path))[0]
    return dict_transkrip.get(audio_id, "").lower()

In [5]:
print(dict_transkrip)

{'00079b49f9': 'Hermes punika ugi dados utusanipun déwa Zeus', '002762f6a1': 'Lakhsmi garwanipun Wisnu punika minangka dewi kemakmuran saha kebahagiaan', '0034ac8d52': 'Kisah pepanggihan kalihan para Siren kacariyosaken ing kisah Odisseia', '00537fd57f': 'Muhammad Iqbal dolan menyang Cirebon', '005a3b3ad4': 'Stephen Chow lunga dhateng Finlandia', '0064a00864': 'Siti tansah eling sakjroning Enno Lerian', '00e1f837f4': 'Kristen Bell lunga dhateng Belfast', '011e017022': 'He moved at glacial speed towards the exam room', '0124f3cd94': 'Krabs rikala iku', '01a357ef3c': 'Rihanna lunga dhateng latvia', '01d15fd955': 'Saiki Kediri wis soyo maju ngoyak daerah liyane', '01e138192f': 'Wedhus kang biyasané ana ing kampung kampung ya iku wedhus kacukan', '01fbf0625b': 'Stephen Chow lunga dhateng Seoul', '020a55f3b6': 'Seogang utawi Kali Kulon mili saking sisih kulon', '020b073967': 'Emma Thompson kuwi alumni Cambridge University', '021d7997a3': 'Ronan Keating kuwi alumni University of Edinburgh', 

In [ ]:
import os
import time
import numpy as np
import pandas as pd

# Path root folder audio (WAV/FLAC)
SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"

# Path CSV transkrip hasil stratified sampling
CSV_PATH = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/"
    r"samplingStratified/out_samplingStratified/dataSampledSmall.csv"
)

# Folder output hasil
DIR_SAVE = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/modelling/"
    r"voiceModelling/srcVoiceModelling/GMM_TDNN/out/GMM"
)

N_COMPONENTS = 16      # K komponen GMM per kata (akan di-tune juga)
N_ITER       = 100     # maks iterasi EM
TOL          = 1e-4    # toleransi konvergensi
N_GRAM       = 3       # orde N-gram (trigram, sesuai sub-bab 3.3.7)
TOP_K_DECODE = 10      # top-K kandidat kata untuk N-gram rescore

JALANKAN_GLOBAL_GMM = True    
JALANKAN_TUNING     = True   

def main():
    os.makedirs(DIR_SAVE, exist_ok=True)
    t_start = time.time()
    print("GMM ASR BAHASA JAWA")
    # TAHAP 1 — LOAD TRANSKRIP
    print("\n[TAHAP 1] Baca Transkrip")
    dict_transkrip  = GMM_pipeline.baca_transkrip_csv(CSV_PATH)
    semua_transkrip = list(dict_transkrip.values())
    print(f" Total audio dengan transkrip: {len(dict_transkrip)}")

    lexicon = buat_lexicon(semua_transkrip)
    print(f" Vocab kata unik : {len(lexicon)}")
    pd.DataFrame([
        {"Kata": k, "Fonem": " ".join(v)} for k, v in lexicon.items()
    ]).to_csv(f"{DIR_SAVE}/HasilLexicon.csv", index=False)

    # TAHAP 2 — EXTRACT MFCC 
    print("\n[TAHAP 2] Ekstraksi MFCC")
    cache_mfcc, meta_list = GMM_pipeline.build_mfcc_cache(
        sampled_root    = SAMPLED_ROOT,
        dict_transkrip  = dict_transkrip,
        verbose         = True,
    )
    df_mfcc = pd.DataFrame(meta_list)
    df_mfcc.to_csv(f"{DIR_SAVE}/HasilMFCC.csv", index=False)
    print(f"  HasilMFCC.csv → {len(df_mfcc)} audio")

    if len(cache_mfcc) == 0:
        print("  [ERROR] Cek path SAMPLED_ROOT.")
        return

    # TAHAP 3 — GMM GLOBAL
    if JALANKAN_GLOBAL_GMM:
        print("\n[TAHAP 3]GMM Global")
        X_global = np.vstack(list(cache_mfcc.values()))
        print(f"  Shape X_global: {X_global.shape}")

        gmm_global = GaussianMixtureModel(
            n_components = N_COMPONENTS,
            n_iter       = N_ITER,
            tol          = TOL,
            reg_covar    = 1e-6,
            random_state = 42,
        )
        gmm_global.fit(X_global)

        pd.DataFrame([{
            "K Komponen"          : N_COMPONENTS,
            "Iterasi EM"          : gmm_global.n_iter_done_,
            "Konvergen"           : gmm_global.converged_,
            "Log-Likelihood Akhir": round(gmm_global.log_likelihood_history_[-1], 6),
        }]).to_csv(f"{DIR_SAVE}/HasilGMMGlobal.csv", index=False)

        print(f"  Konvergen     : {gmm_global.converged_}")
        print(f"  Iterasi EM    : {gmm_global.n_iter_done_}")
        print(f"  Log-Likelihood: {gmm_global.log_likelihood_history_[-1]:.4f}")
    else:
        print("\n[TAHAP 3]GLOBAL_GMM=False")

    # TAHAP 4 — KUMPULKAN FRAME PER KATA
    print("\n[TAHAP 4] Mengumpulkan frame per kata")
    frame_per_kata = func.kumpulkan_frame_per_kata_dari_cache(
        cache_mfcc    = cache_mfcc,
        dict_transkrip= dict_transkrip,
        verbose       = True,
    )

    # TAHAP 5 — LATIH GMM PER KATA
    print(f"\n[TAHAP 5] Melatih GMM per kata (K={N_COMPONENTS})")
    gmm_per_kata = func.latih_gmm_per_kata(
        frame_per_kata = frame_per_kata,
        n_components   = N_COMPONENTS,
        n_iter         = N_ITER,
        tol            = TOL,
        verbose        = True,
    )

    # Simpan ringkasan training GMM per kata
    hasil_training = []
    for kata, gmm in gmm_per_kata.items():
        frame_total = np.vstack(frame_per_kata[kata]).shape[0]
        hasil_training.append({
            "Kata"                : kata,
            "Jumlah Frame"        : frame_total,
            "K Komponen"          : gmm.n_components,
            "Iterasi EM"          : gmm.n_iter_done_,
            "Konvergen"           : gmm.converged_,
            "Log-Likelihood Akhir": round(gmm.log_likelihood_history_[-1], 4),
        })
    pd.DataFrame(hasil_training).to_csv(
        f"{DIR_SAVE}/HasilGMMPerKata.csv", index=False)
    print(f"  HasilGMMPerKata.csv → {len(hasil_training)} kata")

    # TAHAP 6 — LATIH N-GRAM LANGUAGE MODEL
    print(f"\n[TAHAP 6] Melatih {N_GRAM}-Gram Language Model")
    ngram_model = NGramModel(n=N_GRAM, discount=0.75)
    ngram_model.fit(semua_transkrip)
    print(f"  Vocab size : {len(ngram_model.vocab)}")
    print(f"  N-gram unik: {len(ngram_model.ngram_count)}")

    # TAHAP 7 — DEKODE + WER
    print(f"\n[TAHAP 7] Dekode audio + hitung WER ...")
    hasil_wer = func.dekode_semua_audio(
        cache_mfcc    = cache_mfcc,
        dict_transkrip= dict_transkrip,
        gmm_per_kata  = gmm_per_kata,
        ngram_model   = ngram_model,
        top_k         = TOP_K_DECODE,
        verbose       = True,
    )

    df_wer = pd.DataFrame(hasil_wer)
    df_wer.to_csv(f"{DIR_SAVE}/HasilWER_GMM.csv", index=False)
    print(f"  HasilWER_GMM.csv → {len(df_wer)} audio")

    stat_wer = func.ringkasan_wer(df_wer, label="GMM per Kata")
    pd.DataFrame([stat_wer]).to_csv(f"{DIR_SAVE}/HasilRingkasanWER.csv", index=False)

    # TAHAP 8 — HYPERPARAMETER TUNING
    if JALANKAN_TUNING:
        print(f"\n[TAHAP 8] Hyperparameter Tuning")
        df_tuning = func.hyperparameter_tuning(
            frame_per_kata  = frame_per_kata,   
            cache_mfcc      = cache_mfcc,       
            dict_transkrip  = dict_transkrip,
            ngram_model     = ngram_model,
            dir_save        = DIR_SAVE,
            n_gram          = N_GRAM,
        )
    else:
        print("\n[TAHAP 8]TUNING=False")

    total_time = time.time() - t_start
    print("\n" + "=" * 65)
    print(f"  SELESAI dalam {total_time/60:.1f} menit")
    print(f"  Output tersimpan di: {DIR_SAVE}")
    print("=" * 65)


if __name__ == "__main__":
    main()